This is a attempt to train a predictive control using reinforcement learning.

## Setup Overview

### Agent: 

The point absorber. It records its motion (displacements) and takes actions (controls the dampning control).

### Environment: 

This is the cummins equation simulator with the randomly generated jonswap sea state

### Rewards: 

The mean power absorbed over medium time period 

Requirements: run from wave_rl environment as capytaine_env was using a to modern python for stable baselines

In [17]:
import os # for saving model
import gymnasium # going to customise a openai gym environment
from gymnasium import Env 
from gymnasium.spaces import Discrete, Box, Dict, Tuple, MultiBinary, MultiDiscrete
from stable_baselines3 import PPO # built in training model
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.evaluation import evaluate_policy

In [16]:
import numpy as np

In [15]:
from Functions import generate_frequencies, jonswap_frequency_amplitudes, generate_buoy, solve_with_capytaine, get_cummins_components, solve_cummins_stepwise_no_latch_rl, rk4_step

Model Parameters:

In [ ]:
SEED = 42
TRUN = 120
NFREQ = 50
PEAK_PERIOD = 8.0
HS = 2.0
BUOY_MASS = 1000.0
BUOY_RADIUS = 2.0
WATER_DENSITY = 1025.0
WATER_DEPTH = np.inf  # or a finite depth if you needed
WAVE_DIRECTION = 0.0
WAVE_AMPLITUDE = None  # only if you still use it somewhere
C_PTO = 1e5
K_PTO = 0.0
DT = 0.1
RL_STEP_SIZE = 0.25
OBS_HISTORY_SECONDS = 10.0


Modified Functions For RL

In [18]:
# solve with capytaine once as this is the slow part

buoy = generate_buoy(radius=BUOY_RADIUS, mass=BUOY_MASS)

omegas, delta_omega = generate_frequencies(N=NFREQ, Tp=PEAK_PERIOD)

capytaine_dataset = solve_with_capytaine(body=buoy, omegas=omegas, wave_direction=WAVE_DIRECTION, water_depth=WATER_DEPTH, water_density=WATER_DENSITY)

wave_amplitudes = jonswap_frequency_amplitudes(omegas, delta_omega, Hs = 2.0, Tp=12.0 , gamma=3.3) 


Initialising function: generate_buoy
Initialising function: solve_with_capytaine


c:\Users\edwar\anaconda3\envs\wave_rl\Lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for 
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

In [19]:
class WaveEnv(Env):
    """Pass ``buoy``, ``capytaine_dataset``, ``wave_amplitudes``, ``omegas`` from the Capytaine setup cell so the env does not rely on notebook globals."""

    def __init__(self, buoy, capytaine_dataset, wave_amplitudes, omegas, seed=SEED):
        self.buoy = buoy
        self.step_size = RL_STEP_SIZE
        self.run_duration = TRUN
        self.observable_data_points = int(OBS_HISTORY_SECONDS / DT)
        self.c_pto = C_PTO
        self.K_pto = K_PTO
        self.Hs = HS

        self.A_heave_inf, self.t_kernel, self.kernel, self.K_heave, self.F_ex_time, _ = get_cummins_components(
            body=self.buoy, capytaine_dataset=capytaine_dataset, wave_direction=WAVE_DIRECTION,
            wave_amplitudes=wave_amplitudes, omegas=omegas, seed=seed
        )

        self.action_space = Box(low=-1, high=1, shape=(1,))

        # FIX 5: flatten obs to 1D for MlpPolicy
        self.observation_space = Box(low=-np.inf, high=np.inf, shape=(3 * self.observable_data_points,))

        self.history = {
            't':    np.array([0.0], dtype=float),
            'x':    np.array([0.0], dtype=float),
            'v':    np.array([0.0], dtype=float),
            'F_ex': [float(self.F_ex_time(0.0))],
            'c_pto': [float(self.c_pto)],
        }
        self.t_sim = 0.0

    def _get_obs(self):
        obs = np.zeros((3, self.observable_data_points), dtype=np.float32)
        x_hist = np.asarray(self.history['x'], dtype=np.float32)
        v_hist = np.asarray(self.history['v'], dtype=np.float32)
        c_hist = np.asarray(self.history['c_pto'], dtype=np.float32)
        obs[0, -len(x_hist[-self.observable_data_points:]):] = x_hist[-self.observable_data_points:]
        obs[1, -len(v_hist[-self.observable_data_points:]):] = v_hist[-self.observable_data_points:]
        obs[2, -len(c_hist[-self.observable_data_points:]):] = c_hist[-self.observable_data_points:]
        return obs.flatten()  # FIX 5: return 1D

    def step(self, action):
        action = float(np.asarray(action).reshape(-1)[0])

        # FIX 2: clamp c_pto to non-negative
        self.c_pto = max(0.0, self.c_pto + action)

        self.history = solve_cummins_stepwise_no_latch_rl(
            rk4_step, self.buoy, self.A_heave_inf, self.t_kernel, self.kernel,
            self.K_heave, self.F_ex_time, self.c_pto, self.K_pto,
            self.step_size, dt=DT, history=self.history
        )
        self.history['c_pto'].append(float(self.c_pto))

        reward = self.reward()
        self.t_sim += self.step_size
        terminated = self.t_sim >= self.run_duration
        truncated = False

        return self._get_obs(), reward, terminated, truncated, {}

    def reward(self):
        v_hist = np.asarray(self.history['v'], dtype=float)
        n_pts = max(1, int(self.step_size / DT))
        v_seg = v_hist[-n_pts:]
        p_t = float(self.c_pto) * v_seg ** 2
        return float(np.mean(p_t) / self.Hs**2 - float(self.c_pto) ** 2 / 1e5)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.c_pto = float(C_PTO)
        self.t_sim = 0.0
        self.history = {
            't':    np.array([0.0], dtype=float),
            'x':    np.array([0.0], dtype=float),
            'v':    np.array([0.0], dtype=float),
            'F_ex': [float(self.F_ex_time(0.0))],
            'c_pto': [float(self.c_pto)],
        }
        obs = np.zeros((3, self.observable_data_points), dtype=np.float32)
        obs[2, -1] = float(self.c_pto)
        return obs.flatten(), {}  # FIX 5: return 1D


## Train PPO

**Note:** Each environment step runs the Cummins solver for `RL_STEP_SIZE` s of simulated time, so `total_timesteps` is expensive — start small, then scale up.


In [23]:
# Training hyperparameters (tune after a first successful run)
RL_TOTAL_TIMESTEPS = [50000,50000,50000,50000,50000,50000,50000,50000,50000,50000]  # increase for better policies (e.g. 200k+)
RL_N_STEPS = 1024            # steps collected per env before each PPO update (multiples of ~episode length are fine)
RL_BATCH_SIZE = 64
RL_N_EPOCHS = 10
RL_LEARNING_RATE = 3e-4

# Saves next to the working directory when you run the notebook (usually timedomain/)
MODEL_DIR = os.path.abspath(os.path.join(os.getcwd(), "rl_models"))
os.makedirs(MODEL_DIR, exist_ok=True)

In [24]:
# Vector env (required by Stable-Baselines3). One copy of the wave env — add more lambdas to parallelize with SubprocVecEnv later.
train_env = DummyVecEnv([lambda: WaveEnv(buoy, capytaine_dataset, wave_amplitudes, omegas)])

model = PPO(
    "MlpPolicy",
    train_env,
    learning_rate=RL_LEARNING_RATE,
    n_steps=RL_N_STEPS,
    batch_size=RL_BATCH_SIZE,
    n_epochs=RL_N_EPOCHS,
    seed=SEED,
    verbose=1,
)

for i, batch in enumerate(RL_TOTAL_TIMESTEPS):
    model.learn(total_timesteps=batch, progress_bar=False)
    MODEL_PATH = os.path.join(MODEL_DIR, f"ppo_wave_absorber-{i+1}")
    model.save(MODEL_PATH)
    print(f"Saved policy to {MODEL_PATH}.zip")

Initialising function: get_cummins_components
Using cpu device
-----------------------------
| time/              |      |
|    fps             | 976  |
|    iterations      | 1    |
|    time_elapsed    | 1    |
|    total_timesteps | 1024 |
-----------------------------
------------------------------------------
| time/                   |              |
|    fps                  | 792          |
|    iterations           | 2            |
|    time_elapsed         | 2            |
|    total_timesteps      | 2048         |
| train/                  |              |
|    approx_kl            | 1.542503e-08 |
|    clip_fraction        | 0            |
|    clip_range           | 0.2          |
|    entropy_loss         | -1.42        |
|    explained_variance   | -1.19e-07    |
|    learning_rate        | 0.0003       |
|    loss                 | 1.3e+12      |
|    n_updates            | 10           |
|    policy_gradient_loss | -2.68e-06    |
|    std                  | 1          

In [ ]:
# Requires `tqdm` for progress_bar; set progress_bar=False if not installed
model.learn(total_timesteps=RL_TOTAL_TIMESTEPS, progress_bar=False)

model.save(MODEL_PATH)
print(f"Saved policy to {MODEL_PATH}.zip")

## Load Model

In [10]:
# Requires Capytaine setup cell (``buoy``, ``capytaine_dataset``, etc.) to have been run in this session.
model = PPO.load(
    MODEL_PATH,
    env=DummyVecEnv([lambda: WaveEnv(buoy, capytaine_dataset, wave_amplitudes, omegas)]),
)

NameError: name 'buoy' is not defined

### Evaluate trained policy


In [ ]:
eval_env = DummyVecEnv([lambda: WaveEnv(buoy, capytaine_dataset, wave_amplitudes, omegas)])
mean_r, std_r = evaluate_policy(model, eval_env, n_eval_episodes=5, deterministic=True)
print(f"Eval mean reward (5 episodes): {mean_r:.4f} ± {std_r:.4f}")


### Load a saved checkpoint (optional)

Run after restart without retraining: run imports, parameters, Capytaine setup (defines `buoy`, `capytaine_dataset`, `wave_amplitudes`, `omegas`), and the `WaveEnv` class cell; set `MODEL_PATH`; then load as below.


In [ ]:
# model = PPO.load(MODEL_PATH, env=DummyVecEnv([lambda: WaveEnv(buoy, capytaine_dataset, wave_amplitudes, omegas)]))
